# Non-parametric stats on BAPS tables

Converted from `statistics.py`. Run cells in order. The last cell starts the interactive scorer / plots.

Data live under `experiments/LevelN/Results_CSV` and `Results_xdf` (not in git). Outputs go to `analysis/outputs/`.


In [ ]:
# Run from repo root or from analysis/. This replaces __file__ used by the .py scripts.
from pathlib import Path
import os, sys
cwd = Path.cwd().resolve()
if (cwd / "analysis").is_dir() and (cwd / "experiments").is_dir():
    REPO_ROOT = cwd
elif cwd.name == "analysis":
    REPO_ROOT = cwd.parent
elif cwd.name == "notebooks":
    REPO_ROOT = cwd.parents[1]
else:
    REPO_ROOT = cwd
_ANALYSIS_DIR = REPO_ROOT / "analysis"
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)
%matplotlib inline


Statistical analysis for AttentionSpan BAPS tables.

Looks for BAPS_Subject*_results.csv under analysis/outputs, experiments/,
or the local GoNoGoTests/ data dump. Does not run on import.


In [ ]:
import os
import glob
import re
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

REPO = os.path.abspath(os.path.join(str(_ANALYSIS_DIR), ".."))
SEARCH_DIRS = [
    os.environ.get("ATTENTIONSPAN_OUTPUT", os.path.join(REPO, "analysis", "outputs")),
    os.environ.get("ATTENTIONSPAN_DATA", os.path.join(REPO, "experiments")),
    os.path.join(REPO, "GoNoGoTests"),
]

METRICS = ["BAPS_s", "BAPS_3CC", "Recovery_s", "Trials_to_3CC"]
CONDS = ["Baseline", "Distraction+Response", "Distraction+NoResponse"]
COND_ABB = {
    "Baseline": "BL",
    "Distraction+Response": "DR",
    "Distraction+NoResponse": "DN",
}
LEVELS = [1, 2, 3]


def load_baps_tables(search_dirs=None):
    dirs = search_dirs or SEARCH_DIRS
    files = []
    for d in dirs:
        files.extend(glob.glob(os.path.join(d, "BAPS_Subject*_results.csv")))
    files = sorted(set(files))
    if not files:
        return None
    dfs = []
    for f in files:
        sid = int(re.search(r"Subject(\d+)", os.path.basename(f)).group(1))
        df = pd.read_csv(f)
        df["SID"] = sid
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)


def cohen_d_paired(a, b):
    diff = np.array(a) - np.array(b)
    sd = np.std(diff, ddof=1)
    return np.mean(diff) / sd if sd > 0 else np.nan


def sig_stars(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def collect(frame, metric, cond, level=None):
    rows = frame[frame["Condition"] == cond]
    if level is not None:
        rows = rows[rows["Level"] == level]
    return rows.groupby("SID")[metric].mean().dropna()


def condition_effects(frame):
    rows = []
    for level in LEVELS:
        for metric in METRICS:
            bl = collect(frame, metric, "Baseline", level)
            dr = collect(frame, metric, "Distraction+Response", level)
            dn = collect(frame, metric, "Distraction+NoResponse", level)
            sids = bl.index.intersection(dr.index).intersection(dn.index)
            if len(sids) < 3:
                continue
            bl_v, dr_v, dn_v = bl[sids].values, dr[sids].values, dn[sids].values
            stat_f, p_f = stats.friedmanchisquare(bl_v, dr_v, dn_v)
            rows.append({
                "level": level,
                "metric": metric,
                "n": len(sids),
                "BL_mean": bl_v.mean(),
                "DR_mean": dr_v.mean(),
                "DN_mean": dn_v.mean(),
                "friedman_chi2": stat_f,
                "friedman_p": p_f,
                "d_BL_DR": cohen_d_paired(bl_v, dr_v),
                "d_BL_DN": cohen_d_paired(bl_v, dn_v),
                "d_DR_DN": cohen_d_paired(dr_v, dn_v),
            })
    return pd.DataFrame(rows)


In [ ]:
def run_all(frame=None, verbose=True):
    if frame is None:
        frame = load_baps_tables()
    if frame is None:
        print("No BAPS_Subject*_results.csv files found. Run behavioral BAPS first.")
        return None
    table = condition_effects(frame)
    if verbose:
        print(f"Loaded {frame['SID'].nunique()} subjects, {len(frame)} rows.")
        if not table.empty:
            print(table.round(3).to_string(index=False))
    return table


In [ ]:
# Run this cell to execute the script
result = run_all()
if result is None:
    raise SystemExit(1)
